d13_s2_a2.ipynb

In [ ]:
import numpy as np

from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


# Step 1: Generate Dataset
X, y = make_classification(
    n_samples=500,
    n_features=20,
    n_informative=8,
    n_redundant=4,
    class_sep=1.5,
    random_state=42
)


# Step 2: Split Dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=0
)


# Step 3: Build Pipeline
def build_pipeline():

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000))
    ])

    return pipe


# Step 4: Define Parameter Grid
def param_grid():

    return {
        "clf__C": [0.01, 0.1, 1.0, 10.0]
    }


# Step 5: Perform Grid Search
def grid_search(X, y, k=5):

    pipeline = build_pipeline()

    cv = StratifiedKFold(
        n_splits=k,
        shuffle=True,
        random_state=0
    )

    gs = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid(),
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
        refit=True
    )

    gs.fit(X, y)

    return gs


# Step 6: Get Best C Value
def best_c(gs):

    return float(gs.best_params_["clf__C"])


# Step 7: Evaluate Test Accuracy
def test_accuracy(gs, X_test, y_test):

    predictions = gs.best_estimator_.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    return round(float(accuracy), 3)


# Step 8: Run Grid Search
gs = grid_search(X_train, y_train, 5)


# Step 9: Display Results
print("Best C:", best_c(gs))

print("Best CV Score:", round(gs.best_score_, 3))

print("Test Accuracy:", test_accuracy(gs, X_test, y_test))


# Step 10: Display All Grid Search Results
print("\nGrid Search Results:")

for params, score in zip(
    gs.cv_results_["params"],
    gs.cv_results_["mean_test_score"]
):
    print(params, "->", round(score, 3))